# Roman Research Nexus Catalog Database access

******

## Kernel Information and Read-Only Status

To run this notebook, please select the "Roman Research Nexus" kernel at the top right of your window.

This notebook is read-only. You can run cells and make edits, but you must save changes to a different location. We recommend saving the notebook within your home directory, or to a new folder within your home (e.g. <span style="font-variant:small-caps;">file > save notebook as > my-nbs/nb.ipynb</span>). Note that a directory must exist before you attempt to add a notebook to it.

## Imports
Here we import the required packages for our catalog database access examples including:
- *astroquery.mast Catalogs* for accessing and querying Roman catalogs
- *pyvo* for accessing and querying catalogs via MAST's TAP service (Table Access Protocol)
- *astropy.coordinates SkyCoord* for defining sky coordinates
- *astropy.units* to specify angular units
- *matplotlib.pyplot* for simple data visualization
- *numpy* for array comparisons

In [ ]:
from astroquery.mast import Catalogs
import pyvo as vo

from astropy.coordinates import SkyCoord
import astropy.units as u

import matplotlib.pyplot as plt
import numpy as np

# # suppress unimportant unit warnings from many TAP services
# import warnings
# warnings.filterwarnings("ignore", module="pyvo.io.vosi.vodataservice")

******

## Introduction
This notebook demonstrates how to access and query Roman catalogs held at MAST, through (1) astroquery as well as (2) advanced query patterns using Astronomical Data Query Language (ADQL) queries via a Table Access Protocol (TAP) service at MAST.

MAST will provide access to a range of Roman catalog products (such as multiband photometric source catalogs as detected from WFI imaging; spectral metadata catalogs derived from WFI grism and prism spectroscopy; and microlensing event variability and light curve catalogs). This notebook demonstrates methods of querying these catalogs from MAST databases, enabling efficient filtering and sample selection across both spatial location and miriad other properties.

### Defining terms
- *SQL*: Structured Query Language, a programming language used for relational databases.
- *ADQL*: Astronomical Data Query Language, a SQL-like programming language that includes functionality for spatial searches.
- *TAP*: Table Access Protocol, an International Virtual Observatory standard for providing programmatic access to catalogs.

******

## Accessing Roman Catalogs
In this section, we will demonstrate how to obtain information about the catalog schemas (covering descriptions of tables, as well as column units, datatypes, and descriptions), and then methods of connecting to and querying Roman catalogs.

### Obtaining information about Roman Catalogs

Roman catalogs are Level 4 (L4) extracted products, and are produced by multiple Roman data pipelines. For full details about the data processing and detailed contents of Roman catalogs, please see [RDox](https://roman-docs.stsci.edu/data-handbook/wfi-data-levels-and-products#WFIDataLevelsandProducts-Level4-ExtractedDataLevel4).

In the future, user-contributed catalogs (L5) are anticipated to be made available through similar methods; these will be documented separately.

<div class="alert alert-warning" style="color:black; background-color:#ffc5c5; border-color:red;">
    <b style="color:red">Future:</b> Add pointers to integrated schema browser (and also the direct URL access) when available.
</div>

### Querying Roman Catalogs using `astroquery`
To begin, we will query Roman catalogs using `astroquery`. 

For demonstration purposes (as Roman catalogs are not yet available), our example will query the PanSTARRS DR2 catalog near NGC 6535. In addition to a cone search, we will also filter to select only bright, blue stars. 
The specific selection criteria we will apply are:
* Located within 1.5 arcminutes of the NGC 6535
* Are point sources (not extended objects)
* Detected flux in the `g`, `i`, `y` bands (quality cuts)
* Have `i` band magnitudes brighter than 18
* Have `g-y` colors less than 0.8  (the bluest, reddest filters for PanSTARRS)

From the [PanSTARRS DR2 catalogs documentation](https://outerspace.stsci.edu/spaces/PANSTARRS/pages/298812351/PS1+Source+extraction+and+catalogs), 
we see the `MeanObject` table contains the relevant photometric information (note: `astroquery` provides the join of `MeanObject` and `ObjectThin` tables, to provide position information).

With `astroquery`, we can directly filter for some of these constraints, but will need to post-process the returned results to apply other constraints.

In [ ]:
# Resolve the position of NGC 6535
c0 = SkyCoord.from_name("NGC 6535")
c0

In [ ]:
# Query MAST for matching observations in 
# the PanSTARRS DR2 Mean catalog
results_aq = Catalogs.query_criteria(
    catalog="Panstarrs",
    data_release="dr2",
    table="mean",
    coordinates=c0,
    radius = 1.5*u.arcmin,
    columns=["objID", "raMean","decMean",
             "gMeanPSFMag","iMeanPSFMag","yMeanPSFMag",
             "iMeanKronMag"],    # List of columns to return
    gMeanPSFMag=[("gt",-999)],   # mag > -999; missing magnitudes are -999
    iMeanPSFMag=[("gt",-999)],   # mag > -999; missing magnitudes are -999
    yMeanPSFMag=[("gt",-999)],   # mag > -999; missing magnitudes are -999
    iMeanKronMag=[("gt",-999)],  # mag > -999; missing magnitudes are -999
)

# Post-process cuts:
# 1. Distinguish stars from extended objects using
#    difference of PSF, Kron magnitudes
results_aq = results_aq[
    (results_aq['iMeanPSFMag'] - results_aq['iMeanKronMag']) <= 0.05
]

# 2. Apply color, magnitude cuts
# Copy original catalog for reference:
results_aq_all = results_aq.copy()
results_aq = results_aq[
    ((results_aq['gMeanPSFMag'] - results_aq['yMeanPSFMag']) < 0.8) 
    & (results_aq['iMeanPSFMag'] < 18)
]

# Final sample selection
results_aq

We will quickly visualize our results by plotting a 
$g-y$ vs $i$ color-magnitude diagram.

In [ ]:
f, ax = plt.subplots()
f.set_size_inches((5,5))

# Stars meeting the color/magnitude cuts
ax.scatter(
    results_aq['gMeanPSFMag']-results_aq['yMeanPSFMag'],
    results_aq['iMeanPSFMag'], 
    s=15,lw=0, color='red',
    label='Bright blue stars',
)

# Plot the full set of stars for comparison
ax.scatter(
    results_aq_all['gMeanPSFMag']-results_aq_all['yMeanPSFMag'],
    results_aq_all['iMeanPSFMag'], 
    s=5,lw=0,zorder=-1, color='gray',
    label='All stars in region',
)

ax.legend()

ax.set_xlabel("g-y, PSF Mag")
ax.set_ylabel("i, PSF Mag")

ax.set_ylim(ax.get_ylim()[::-1])    # Invert for correct magnitude order

### Querying Roman Catalogs using MAST's TAP service using `pyvo`
It is also possible to access Roman catalogs via a TAP service at MAST, by submitting queries written in ADQL. A benefit of constructing queries in the SQL-like language ADQL is that we can specify advanced query patters, such as comparison constraints and derived constraints (i.e., as needed to apply the star/resolved and color criteria in this example), directly in the query. This makes it possible get the sample of interest in one query, avoiding a two step process where we load a larger sample into memory and then perform final cuts in our notebook.

For this second demonstration, we will apply the same selection criteria as above to select bright, blue stars near NGC 6535 from the PanSTARRS DR2 catalog.

First, we connect to the MAST PanSTARRS DR2 TAP service.
<!-- 
We will now repeat the same 



directly get results on our objects of interest, and 

To begin, we will query Roman catalogs using `astroquery`. 

For demonstration purposes (as Roman catalogs are not yet available), our example will query the PanSTARRS DR2 catalog near NGC 6535. In addition to a cone search, we will also filter to select only bright, blue stars. 
The specific selection criteria we will apply are:
* Located within 1.5 arcminutes of the NGC 6535
* Are point sources (not extended objects)
* Detected flux in the `g`, `i`, `y` bands (quality cuts)
* Have `i` band magnitudes brighter than 18
* Have `g-y` colors less than 0.8  (the bluest, reddest filters for PanSTARRS)

From the [PanSTARRS DR2 catalogs documentation](https://outerspace.stsci.edu/spaces/PANSTARRS/pages/298812351/PS1+Source+extraction+and+catalogs), 
we see the `MeanObject` table contains the relevant photometric information (note: `astroquery` provides the join of `MeanObject` and `ObjectThin` tables, to provide position information).

With `astroquery`, we can directly filter for some of these constraints, but will need to post-process the returned results for some others. -->

In [ ]:
# Use `pyvo` to connect to the MAST PanSTARRS TAP service:
TAP_service = vo.dal.TAPService(
    "https://mast.stsci.edu/vo-tap/api/v0.1/ps1_dr2/"
)

Again, we will access the mean object catalog (`mean_object` here; see the [PanSTARRS DR2 catalogs documentation](https://outerspace.stsci.edu/spaces/PANSTARRS/pages/298812351/PS1+Source+extraction+and+catalogs)), which contains all the columns we will need.

In [ ]:
# # Note: it is possible to use the TAP service to directly access catalog/table 
# # schemas, which describe the tables and columns. 
# # An example of this is provided here, commented out as we have pre-identified 
# # the relevent columns from the documentation

# # An overview description of methods, capabilities, and 
# # maximum result set sizes is available via:
# TAP_service.describe()

# # Additionally, the `pyvo` TAPService class objects contain 
# # metadata information about the full catalog schema, including tables and columns.
# # It's possible to iterate through the tables and print column names 
# # as in this code snipppet:
# TAP_tables = TAP_service.tables
# for tablename in TAP_tables.keys():
#     # This can be refined by table name to print columns only for subsets of tables.
#     if (tablename.startswith("ps1_dr2.mean_object")):  # Include only PS1 tables 
#         ####
#         # The table descriptions only include an overview, so this also prints the column names
#         TAP_tables[tablename].describe() 
#         print("Columns={}".format(sorted([k.name for k in TAP_tables[tablename].columns])))
#         print("----")

We now construct our query, including the color/magnitude constraints and 
star/extended object separation. Note that for ADQL, the cone search radius must be expressed in degrees.



In [ ]:
# Query MAST for matching observations in 
# the PanSTARRS DR2 Mean catalog
catalog_data = Catalogs.query_criteria(
    catalog="Panstarrs",
    data_release="dr2",
    table="mean",
    coordinates=c0,
    radius = 1.5*u.arcmin,
    columns=["objID", "raMean","decMean",
             "gMeanPSFMag","iMeanPSFMag","yMeanPSFMag",
             "iMeanKronMag"],    # List of columns to return
    gMeanPSFMag=[("gt",-999)],   # mag > -999; missing magnitudes are -999
    iMeanPSFMag=[("gt",-999)],   # mag > -999; missing magnitudes are -999
    yMeanPSFMag=[("gt",-999)],   # mag > -999; missing magnitudes are -999
    iMeanKronMag=[("gt",-999)],  # mag > -999; missing magnitudes are -999
)

# Post-process cuts:
# 1. Distinguish stars from extended objects using
#    difference of PSF, Kron magnitudes
catalog_data = catalog_data[
    (catalog_data['iMeanPSFMag'] - catalog_data['iMeanKronMag']) <= 0.05
]

# 2. Apply color, magnitude cuts
# Copy original catalog for reference:
catalog_data_all = catalog_data.copy()
catalog_data = catalog_data[
    ((catalog_data['gMeanPSFMag'] - catalog_data['yMeanPSFMag']) < 0.8) 
    & (catalog_data['iMeanPSFMag'] < 18)
]

# Final sample selection
catalog_data

In [ ]:
# Query MAST for matching observations in 
# the PanSTARRS DR2 mean catalog

# Note: ADQL comments are preceded by "--"
adql_query = f"""
SELECT objid, ramean, decmean, 
    gmeanpsfmag, imeanpsfmag, ymeanpsfmag, 
    imeankronmag
FROM mean_object
WHERE CONTAINS(
    POINT('ICRS', ramean, decmean),
    CIRCLE('ICRS',{c0.ra.deg}, {c0.dec.deg}, {(1.5*u.arcmin).to_value(u.deg)})          
)=1                                       -- 1.5 arcmin around NGC 6535
AND gmeanpsfmag > -999                    -- mag > -999; missing magnitudes are -999
AND imeanpsfmag > -999                    -- mag > -999; missing magnitudes are -999
AND ymeanpsfmag > -999                    -- mag > -999; missing magnitudes are -999
AND imeankronmag > -999                   -- mag > -999; missing magnitudes are -999
AND (imeanpsfmag - imeankronmag) <= 0.05  -- distinguish stars from extended objects
AND (gmeanpsfmag-ymeanpsfmag) < 0.8       -- color cut
AND imeanpsfmag < 18                      -- magnitude cut
"""

# Submit query to the MAST TAP service:
job = TAP_service.run_async(adql_query)

# Retrieve results:
results_tap = job.to_table()

By leveraging the advanced usage patterns possible with AQDL, this single TAP query returns the same results as our two-step `astroquery` example where we first queried and then applied the final cuts.

(For this simple example, applying such cuts in memory are not prohibitive, but for cases where the selected sample may be large, performing all cuts server-side can be much more performant.)

In [ ]:
# Sort both by objid
results_aq.sort('objID')
results_tap.sort('objid')
# Check the ID lists are the same:
print(np.all(results_aq['objID'] == results_tap['objid']))

*********


## Additional Resources
Additional information can be found at the following links:

### Other RRN notebooks
- [Data Discovery and Access:](../data_discovery_and_access/data_discovery_and_access.ipynb) The next step in this workflow, covering how to search for and access (particularly cloud streaming access) of file-based Roman products.

### MAST notebooks
- [MAST Table Access Protocol PanSTARRS 1 DR2 Demo](https://spacetelescope.github.io/mast_notebooks/notebooks/PanSTARRS/PS1_DR2_TAP/PS1_DR2_TAP.html), providing a more in-depth tutorial about constructing ADQL queries and using MAST TAP services.

### Other resources
- [RDox Level 4 Product Documentation](https://roman-docs.stsci.edu/data-handbook/wfi-data-levels-and-products#WFIDataLevelsandProducts-Level4-ExtractedDataLevel4), including catalog products.
- [`astroquery.mast.Catalogs` search documentation](https://astroquery.readthedocs.io/en/latest/mast/mast_catalog.html)
- [Documentation for `pyvo`](https://pyvo.readthedocs.io/en/latest/index.html), a package to retrieve astronomical data available from archives that support standard IVOA virtual observatory service protocols.
- [Documentation on ADQL](http://www.ivoa.net/documents/latest/ADQL.html), covering the specifications of this IVOA standard for querying astronomical data in tabular format, with geometric search support.
- [Documentation on TAP](http://www.ivoa.net/documents/TAP/), the IVOA standard for RESTful web service access to tabular data.
- [Full list of TAP services at MAST](https://mast.stsci.edu/vo-tap)


### Citations and acknowledgements
* [Citing `astropy`](https://www.astropy.org/acknowledging.html)
* [Acknowledging PanSTARRS](https://archive.stsci.edu/publishing/mission-acknowledgements#section-895d38a0-86b3-4143-b521-6cc3312701f9)
* [Acknowledging MAST](https://archive.stsci.edu/gsc/mast_data_use.html)

******

## About this Notebook
The information about TAP access is based on the TIKE MAST PanSTARRS DR2 TAP notebook by Rick White & Theresa Dower, with edits from T. Dutkiewicz. 

**Author:**  Sedona Price\
**Updated On:** 2026-03-20

<table width="100%" style="border:none; border-collapse:collapse;">
  <tr style="border:none;">
    <td style="border:none; width:180px; white-space:nowrap;">
       <a href="#top" style="text-decoration:none; color:#0066cc;">↑ Top of page</a> 
    </td>
    <td style="border:none; text-align:center;">
       <img src="../../roman_logo.png" width="50">
    </td>
    <td style="border:none; text-align:right;">
       <img src="../../stsci_logo2.png" width="90">
    </td>
  </tr>
</table>